# SafeStep · 8 — Export the app bundle
Collects models, the shared code, demo data and every result table into **`SafeStep/SafeStep_bundle.zip`**.
Download it, unzip next to `app.py` as a folder named `bundle/`, then `streamlit run app.py`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, sys, json, time, shutil, warnings
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score, confusion_matrix, average_precision_score, balanced_accuracy_score
warnings.filterwarnings('ignore')
BASE = '/content/drive/MyDrive/Daffodil Robo Tech Olympiad'
PROJ = f'{BASE}/SafeStep'
D = {k: f'{PROJ}/{k}' for k in ['data', 'artifacts', 'figures', 'models', 'code']}
DEPLOY = f"{D['models']}/deploy"; os.makedirs(DEPLOY, exist_ok=True)
sys.path.insert(0, D['code']); import safestep_core as core
SEED = 42; rng = np.random.default_rng(SEED)
sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 110
def save_fig(name): plt.savefig(f"{D['figures']}/{name}.png", dpi=300, bbox_inches='tight')
FS, W = core.FS, core.WIN
F    = pd.read_parquet(f"{D['data']}/features_345.parquet").set_index('win_id')
wm   = pd.read_parquet(f"{D['data']}/windows_meta.parquet").set_index('win_id')
meta = pd.read_parquet(f"{D['data']}/trials_meta.parquet").set_index('trial_id')
SEL  = json.load(open(f"{D['artifacts']}/feature_selection.json"))
CFG  = json.load(open(f"{DEPLOY}/deploy_config.json"))
try:
    import lightgbm as lgb
    def make_model(): return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                                                colsample_bytree=0.8, class_weight='balanced', n_jobs=-1, random_state=SEED, verbose=-1)
except ImportError:
    def make_model(): return ExtraTreesClassifier(n_estimators=400, min_samples_leaf=2, class_weight='balanced', n_jobs=-1, random_state=SEED)
print(F.shape, '| deployed fall model:', CFG['fall_model'], '| activity model:', CFG['activity_model'])

(136051, 345) | deployed fall model: MLP | activity model: LightGBM


In [3]:
OUT = f'{PROJ}/app_bundle'
if os.path.exists(OUT): shutil.rmtree(OUT)
for sub in ['code', 'models', 'data', 'results', 'demo_phone']: os.makedirs(f'{OUT}/{sub}')
shutil.copy(f"{D['code']}/safestep_core.py", f'{OUT}/code/')
for f in os.listdir(DEPLOY): shutil.copy(f'{DEPLOY}/{f}', f'{OUT}/models/')
print('models:', sorted(os.listdir(f'{OUT}/models')))

models: ['activity_model.joblib', 'context_model.joblib', 'deploy_config.json', 'direction_model.joblib', 'explain_config.json', 'explain_model.joblib', 'extra_config.json', 'fall_model.joblib', 'preimpact_model.joblib']


In [4]:
signals = np.load(f"{D['data']}/signals_50hz.npz")['signals']
np.savez_compressed(f'{OUT}/data/signals_50hz_f16.npz', signals=signals.astype(np.float16))
meta.reset_index().to_parquet(f'{OUT}/data/trials_meta.parquet', index=False)

keep = ['trial_id', 'start', 'phase', 'y_activity', 'y_fall', 'code', 'name', 'subject', 'age_group', 'type', 'split']
te = wm[wm.split == 'test'][keep]
pd.concat([te, F.loc[te.index]], axis=1).reset_index().to_parquet(f'{OUT}/data/test_windows.parquet', index=False)

parts = []
for a, g in wm[wm.y_activity != 'ignore'].groupby('y_activity'):
    parts.append(g.sample(min(len(g), 2000), random_state=SEED))
smp = pd.concat(parts)[['y_activity', 'split', 'subject', 'code']]
pd.concat([smp, F.loc[smp.index]], axis=1).reset_index().to_parquet(f'{OUT}/data/features_sample.parquet', index=False)
for f in ['test_stream_probs.parquet', 'test_stream_probs_deep.parquet']:
    if os.path.exists(f"{D['data']}/{f}"): shutil.copy(f"{D['data']}/{f}", f'{OUT}/data/')
print('test windows:', len(te), '| feature sample:', len(smp))

test windows: 35138 | feature sample: 20000


In [5]:
for f in os.listdir(D['artifacts']):
    if f.endswith(('.csv', '.json')): shutil.copy(f"{D['artifacts']}/{f}", f'{OUT}/results/')
os.makedirs(f'{OUT}/results/figures', exist_ok=True)
for f in os.listdir(D['figures']): shutil.copy(f"{D['figures']}/{f}", f'{OUT}/results/figures/')
print(len(os.listdir(f'{OUT}/results')), 'result files')

24 result files


## Demo phone files (test recordings rewritten the way a phone exports them)

In [6]:
def to_phone_csv(tid, path, seed):
    r = meta.loc[tid]; x = signals[r.offset:r.offset + r.n_samples].astype(np.float64)
    rr = np.random.default_rng(seed); ax = rr.normal(size=3); ax /= np.linalg.norm(ax); ang = rr.uniform(0.5, 2.5)
    Kx = np.array([[0, -ax[2], ax[1]], [ax[2], 0, -ax[0]], [-ax[1], ax[0], 0]]); R = np.eye(3) + np.sin(ang) * Kx + (1 - np.cos(ang)) * Kx @ Kx
    t50 = np.arange(len(x)) / FS
    t = np.cumsum(rr.uniform(0.008, 0.012, int(t50[-1] / 0.01)))           # ~100 Hz with jitter
    t = t[t < t50[-1]]
    acc = np.column_stack([np.interp(t, t50, (x[:, :3] @ R.T)[:, i]) for i in range(3)]) * core.G
    gyr = np.radians(np.column_stack([np.interp(t, t50, (x[:, 3:6] @ R.T)[:, i]) for i in range(3)]))
    pd.DataFrame({'time_s': t, 'acc_x_ms2': acc[:, 0], 'acc_y_ms2': acc[:, 1], 'acc_z_ms2': acc[:, 2],
                  'gyro_x_rads': gyr[:, 0], 'gyro_y_rads': gyr[:, 1], 'gyro_z_rads': gyr[:, 2]}).to_csv(path, index=False)
tm = meta[meta.split == 'test']
picks = {'demo_fall_forward.csv': tm[tm.code == 'F01'], 'demo_fall_fainting_seated.csv': tm[tm.code == 'F13'],
         'demo_walking.csv': tm[tm.code == 'D01'], 'demo_stumble_nearfall.csv': tm[tm.code == 'D18'],
         'demo_sit_down_quickly.csv': tm[tm.code == 'D10']}
for i, (name, g) in enumerate(picks.items()):
    if len(g): to_phone_csv(g.index[0], f'{OUT}/demo_phone/{name}', i)
print(sorted(os.listdir(f'{OUT}/demo_phone')))

['demo_fall_fainting_seated.csv', 'demo_fall_forward.csv', 'demo_sit_down_quickly.csv', 'demo_stumble_nearfall.csv', 'demo_walking.csv']


In [7]:
import sklearn, scipy, platform
vers = {'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__, 'joblib': joblib.__version__}
for mod in ['lightgbm', 'xgboost']:
    try: vers[mod] = __import__(mod).__version__
    except ImportError: pass
with open(f'{OUT}/requirements_pinned.txt', 'w') as f:
    f.write('streamlit>=1.38\nplotly>=5.20\npyarrow\n' + ''.join(f'{k}=={v}\n' for k, v in vers.items()))
print(open(f'{OUT}/requirements_pinned.txt').read(), '| python', platform.python_version())
zip_path = shutil.make_archive(f'{PROJ}/SafeStep_bundle', 'zip', OUT)
tot = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(OUT) for f in fs)
print(f'bundle {tot/1e6:.0f} MB → {zip_path} ({os.path.getsize(zip_path)/1e6:.0f} MB)')

streamlit>=1.38
plotly>=5.20
pyarrow
scikit-learn==1.6.1
numpy==2.4.6
pandas==2.2.3
scipy==1.16.3
joblib==1.6.0
lightgbm==4.6.0
xgboost==3.4.1
 | python 3.13.16
bundle 191 MB → /content/drive/MyDrive/Daffodil Robo Tech Olympiad/SafeStep/SafeStep_bundle.zip (181 MB)
